# Phase 28 — Hierarchical Modeling

Δύο προσεγγίσεις που εκμεταλλεύονται τη σχέση hazard→product.

## Προσέγγιση Α — Sequential BERT
```
Text → BERT → predict hazard
Text + hazard_onehot → BERT → predict product
```

## Προσέγγιση Β — Conditional SVM
```
Text → BERT → predict hazard
if hazard == 'biological': use SVM_biological to predict product
if hazard == 'allergens':  use SVM_allergens  to predict product
...
```

**Γιατί hierarchical:** Το product εξαρτάται από το hazard.
Π.χ. biological → meat/dairy, allergens → nuts/cereals

**Best so far:** 0.81728

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import re
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score
import torch.nn.functional as F
import copy
import warnings
warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Device: cpu


In [2]:
train = pd.read_csv('train.csv')
valid = pd.read_csv('valid.csv')
test  = pd.read_csv('test.csv')

def make_text(df):
    return (df['title'].fillna('') + ' ' + df['text'].fillna('').str[:550]).tolist()

texts_train = make_text(train)
texts_valid = make_text(valid)
texts_test  = make_text(test)

le_hazard  = LabelEncoder()
le_product = LabelEncoder()
all_hazard  = pd.concat([train['hazard-category'],  valid['hazard-category']])
all_product = pd.concat([train['product-category'], valid['product-category']])
le_hazard.fit(all_hazard)
le_product.fit(all_product)

n_hazard  = len(le_hazard.classes_)
n_product = len(le_product.classes_)
print(f'Hazard classes:  {n_hazard}')
print(f'Product classes: {n_product}')
print(f'Hazard labels: {list(le_hazard.classes_)}')

Hazard classes:  10
Product classes: 22
Hazard labels: ['allergens', 'biological', 'chemical', 'food additives and flavourings', 'foreign bodies', 'fraud', 'migration', 'organoleptic aspects', 'other hazard', 'packaging defect']


In [3]:
def official_st1_score(y_true_hazard, y_pred_hazard,
                       y_true_product, y_pred_product, verbose=True):
    f1_hazard = f1_score(y_true_hazard, y_pred_hazard, average='macro', zero_division=0)
    mask = (np.array(y_true_hazard) == np.array(y_pred_hazard))
    f1_product = f1_score(
        np.array(y_true_product)[mask],
        np.array(y_pred_product)[mask],
        average='macro', zero_division=0
    ) if mask.sum() > 0 else 0.0
    score = (f1_hazard + f1_product) / 2
    if verbose:
        print(f'  macro-F1 Hazard:                    {f1_hazard:.4f}')
        print(f'  Σωστά hazard predictions:           {mask.sum()}/{len(mask)} ({100*mask.mean():.1f}%)')
        print(f'  macro-F1 Product (given correct h): {f1_product:.4f}')
        print(f'  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
        print(f'  OFFICIAL ST1 SCORE:                 {score:.4f}')
    return score

# ΠΡΟΣΕΓΓΙΣΗ Β — Conditional SVM

**Βήμα 1:** Χρησιμοποιούμε τα ήδη υπάρχοντα BERT-base Focal probs για hazard prediction.

**Βήμα 2:** Για κάθε hazard κλάση, εκπαιδεύουμε ξεχωριστό SVM classifier για product
μόνο στα δείγματα εκείνης της κλάσης.

In [4]:
# Φόρτωση BERT-base Focal hazard predictions
# (από phase15 — train-only model)
bert_valid_hazard_probs = np.load('npy/bertbase_focal_trainonly_valid_hazard_probs.npy')
bert_test_hazard_probs  = np.load('npy/bertbase_focal_test_hazard_probs2.npy')

pred_hazard_valid = le_hazard.inverse_transform(bert_valid_hazard_probs.argmax(axis=1))
pred_hazard_test  = le_hazard.inverse_transform(bert_test_hazard_probs.argmax(axis=1))

print(f'Valid hazard predictions: {len(pred_hazard_valid)}')
print(f'Test hazard predictions:  {len(pred_hazard_test)}')

# Κατανομή predictions
from collections import Counter
print('\nValid hazard distribution:')
for cls, cnt in Counter(pred_hazard_valid).most_common():
    print(f'  {cls:35s}: {cnt}')

Valid hazard predictions: 565
Test hazard predictions:  997

Valid hazard distribution:
  other hazard                       : 364
  foreign bodies                     : 111
  fraud                              : 35
  chemical                           : 29
  packaging defect                   : 10
  organoleptic aspects               : 9
  food additives and flavourings     : 7


In [5]:
# TF-IDF για SVM
def preprocess(text):
    if not isinstance(text, str): return ''
    text = text.lower()
    text = re.sub(r'\d+', ' ', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

train['combined'] = train['title'].apply(preprocess) + ' ' + train['text'].str[:550].apply(preprocess)
valid['combined'] = valid['title'].apply(preprocess) + ' ' + valid['text'].str[:550].apply(preprocess)
test['combined']  = test['title'].apply(preprocess)  + ' ' + test['text'].str[:550].apply(preprocess)

tfidf = TfidfVectorizer(max_features=50000, ngram_range=(1,2),
                         sublinear_tf=True, stop_words='english')
X_train_tfidf = tfidf.fit_transform(train['combined'])
X_valid_tfidf = tfidf.transform(valid['combined'])
X_test_tfidf  = tfidf.transform(test['combined'])

print(f'TF-IDF features: {X_train_tfidf.shape[1]}')

TF-IDF features: 50000


In [7]:
# Εκπαίδευση conditional SVM — ένας classifier ανά hazard κλάση
print('=== ΕΚΠΑΙΔΕΥΣΗ CONDITIONAL SVM ===\n')

conditional_svms = {}
hazard_classes = le_hazard.classes_

for hazard_cls in hazard_classes:
    # Φίλτραρε δείγματα αυτής της κλάσης
    mask = (train['hazard-category'] == hazard_cls).values
    n_samples = mask.sum()

    if n_samples < 5:
        print(f'  {hazard_cls:35s}: SKIP (μόνο {n_samples} δείγματα)')
        conditional_svms[hazard_cls] = None
        continue

    X_cls = X_train_tfidf[mask]
    y_cls = train['product-category'][mask]

    # Αν η κλάση έχει μόνο 1 product κλάση, δεν χρειάζεται classifier
    if y_cls.nunique() == 1:
        print(f'  {hazard_cls:35s}: SINGLE CLASS ({y_cls.iloc[0]})')
        conditional_svms[hazard_cls] = y_cls.iloc[0]  # string, όχι classifier
        continue

    clf = LinearSVC(C=0.5, class_weight='balanced', max_iter=2000, random_state=42)
    clf.fit(X_cls, y_cls)
    conditional_svms[hazard_cls] = clf
    print(f'  {hazard_cls:35s}: trained on {n_samples} samples, {y_cls.nunique()} product classes')

print('\nConditional SVMs εκπαιδεύτηκαν')

=== ΕΚΠΑΙΔΕΥΣΗ CONDITIONAL SVM ===

  allergens                          : trained on 1854 samples, 18 product classes
  biological                         : trained on 1741 samples, 18 product classes
  chemical                           : trained on 287 samples, 20 product classes
  food additives and flavourings     : trained on 24 samples, 7 product classes
  foreign bodies                     : trained on 561 samples, 17 product classes
  fraud                              : trained on 371 samples, 19 product classes
  migration                          : SKIP (μόνο 3 δείγματα)
  organoleptic aspects               : trained on 53 samples, 13 product classes
  other hazard                       : trained on 134 samples, 18 product classes
  packaging defect                   : trained on 54 samples, 13 product classes

Conditional SVMs εκπαιδεύτηκαν


In [8]:
def predict_conditional(pred_hazard, X_tfidf):
    """Για κάθε δείγμα, χρησιμοποιεί τον SVM της αντίστοιχης hazard κλάσης."""
    pred_product = []

    for i, hazard_cls in enumerate(pred_hazard):
        clf = conditional_svms.get(hazard_cls)

        if clf is None:
            # Fallback: global SVM
            pred_product.append('meat, egg and dairy products')  # most common
        elif isinstance(clf, str):
            # Single class
            pred_product.append(clf)
        else:
            # Predict με τον conditional classifier
            pred_product.append(clf.predict(X_tfidf[i])[0])

    return np.array(pred_product)


# Validation evaluation
pred_product_valid_cond = predict_conditional(pred_hazard_valid, X_valid_tfidf)

print('=== ΑΞΙΟΛΟΓΗΣΗ Προσέγγιση Β — Conditional SVM ===')
score_b = official_st1_score(
    valid['hazard-category'], pred_hazard_valid,
    valid['product-category'], pred_product_valid_cond
)

# Test submission
pred_product_test_cond = predict_conditional(pred_hazard_test, X_test_tfidf)

pd.DataFrame({
    'id': test['id'],
    'hazard-category': pred_hazard_test,
    'product-category': pred_product_test_cond
}).to_csv('submission_hierarchical_b.csv', index=False)
print('\nΑποθηκεύτηκε: submission_hierarchical_b.csv')

=== ΑΞΙΟΛΟΓΗΣΗ Προσέγγιση Β — Conditional SVM ===
  macro-F1 Hazard:                    0.4054
  Σωστά hazard predictions:           114/565 (20.2%)
  macro-F1 Product (given correct h): 0.4724
  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  OFFICIAL ST1 SCORE:                 0.4389

Αποθηκεύτηκε: submission_hierarchical_b.csv


# ΠΡΟΣΕΓΓΙΣΗ Α — Sequential BERT

**Βήμα 1:** BERT → predict hazard

**Βήμα 2:** BERT encoder + one-hot(hazard) → predict product

Ο product classifier βλέπει και το κείμενο ΚΑΙ το predicted hazard.


In [ ]:
MODEL_NAME   = 'bert-base-uncased'
BATCH_SIZE   = 32
MAX_LENGTH   = 128
LR           = 2e-5
N_EPOCHS     = 20
WARMUP_RATIO = 0.1

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f'Tokenizer loaded: {MODEL_NAME}')


class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts      = texts
        self.labels     = labels
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoded = self.tokenizer(
            str(self.texts[idx]),
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      encoded['input_ids'].squeeze(),
            'attention_mask': encoded['attention_mask'].squeeze(),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long)
        }


# Dataset για Stage 2 — περιλαμβάνει hazard one-hot
class HierarchicalDataset(Dataset):
    def __init__(self, texts, hazard_labels_onehot, product_labels, tokenizer, max_length=128):
        self.texts               = texts
        self.hazard_labels_onehot = hazard_labels_onehot  # (N, n_hazard)
        self.product_labels      = product_labels
        self.tokenizer           = tokenizer
        self.max_length          = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoded = self.tokenizer(
            str(self.texts[idx]),
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      encoded['input_ids'].squeeze(),
            'attention_mask': encoded['attention_mask'].squeeze(),
            'hazard_onehot':  torch.tensor(self.hazard_labels_onehot[idx], dtype=torch.float),
            'label':          torch.tensor(self.product_labels[idx], dtype=torch.long)
        }

print('Datasets defined')

In [9]:
class FocalLoss(nn.Module):
    """
    Focal Loss για class imbalance.
    FL(p_t) = -alpha_t * (1 - p_t)^gamma * log(p_t)

    gamma=0 → κανονικό Cross Entropy
    gamma=2 → standard Focal Loss (Lin et al. 2017)
    """
    def __init__(self, gamma=2.0, weight=None):
        super().__init__()
        self.gamma  = gamma
        self.weight = weight

    def forward(self, logits, targets):
        ce_loss = F.cross_entropy(logits, targets, weight=self.weight, reduction='none')
        pt      = torch.exp(-ce_loss)          # p_t = probability of correct class
        focal   = (1 - pt) ** self.gamma * ce_loss
        return focal.mean()

In [ ]:
# Stage 2 Model: BERT encoder + hazard one-hot → product
class HierarchicalBERT(nn.Module):
    def __init__(self, model_name, n_hazard, n_product, dropout=0.1):
        super().__init__()
        self.encoder  = AutoModel.from_pretrained(model_name)
        hidden_size   = self.encoder.config.hidden_size  # 768 για BERT-base

        # Classifier: [CLS] (768) + hazard one-hot (n_hazard) → product
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size + n_hazard, n_product)
        )

    def forward(self, input_ids, attention_mask, hazard_onehot):
        outputs    = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]  # [CLS] token

        # Concatenation [CLS] + hazard one-hot
        combined = torch.cat([cls_output, hazard_onehot], dim=1)
        return self.classifier(combined)

print('HierarchicalBERT defined!')
print(f'Input size: 768 (BERT) + {n_hazard} (hazard one-hot) = {768 + n_hazard}')

In [ ]:
# === STAGE 1: Φόρτωση ήδη υπάρχοντος BERT hazard model ===
# Χρησιμοποιούμε τα probs από το phase15 (train-only)
print('=== STAGE 1: Hazard Predictions (από BERT-base Focal) ===')

# Valid hazard predictions (one-hot)
pred_hazard_valid_idx = bert_valid_hazard_probs.argmax(axis=1)
hazard_onehot_valid   = np.eye(n_hazard)[pred_hazard_valid_idx]

# Train hazard predictions — χρησιμοποιούμε true labels για training
# (teacher forcing στο training)
y_train_hazard_idx  = le_hazard.transform(train['hazard-category'])
hazard_onehot_train = np.eye(n_hazard)[y_train_hazard_idx]

y_train_product_idx = le_product.transform(train['product-category'])
y_valid_product_idx = le_product.transform(valid['product-category'])

print(f'Train hazard one-hot shape: {hazard_onehot_train.shape}')
print(f'Valid hazard one-hot shape: {hazard_onehot_valid.shape}')
print('\nΣτο training χρησιμοποιούμε TRUE hazard labels (teacher forcing)')
print('Στο validation/test χρησιμοποιούμε PREDICTED hazard labels')

In [ ]:
# === STAGE 2: Εκπαίδευση HierarchicalBERT για product ===
print('=== STAGE 2: Εκπαίδευση HierarchicalBERT — PRODUCT ===')
print(f'Train: {len(texts_train)} | Valid: {len(texts_valid)}\n')

train_loader = DataLoader(
    HierarchicalDataset(texts_train, hazard_onehot_train, y_train_product_idx, tokenizer, MAX_LENGTH),
    batch_size=BATCH_SIZE, shuffle=True
)
valid_loader = DataLoader(
    HierarchicalDataset(texts_valid, hazard_onehot_valid, y_valid_product_idx, tokenizer, MAX_LENGTH),
    batch_size=BATCH_SIZE, shuffle=False
)

hier_model = HierarchicalBERT(MODEL_NAME, n_hazard, n_product).to(device)
optimizer  = AdamW(hier_model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * N_EPOCHS
scheduler   = get_linear_schedule_with_warmup(
    optimizer, int(total_steps * WARMUP_RATIO), total_steps
)
#criterion = nn.CrossEntropyLoss() 1st try
criterion = FocalLoss(gamma=2.0)

best_f1    = 0
best_state = None
patience   = 3
patience_cnt = 0

for epoch in range(N_EPOCHS):
    hier_model.train()
    total_loss = 0
    for batch in train_loader:
        labels        = batch.pop('label').to(device)
        hazard_onehot = batch.pop('hazard_onehot').to(device)
        batch         = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()
        logits = hier_model(**batch, hazard_onehot=hazard_onehot)
        loss   = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(hier_model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    # Validation
    hier_model.eval()
    all_preds = []
    with torch.no_grad():
        for batch in valid_loader:
            batch.pop('label')
            hazard_onehot = batch.pop('hazard_onehot').to(device)
            batch = {k: v.to(device) for k, v in batch.items()}
            preds = hier_model(**batch, hazard_onehot=hazard_onehot).argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)

    f1 = f1_score(y_valid_product_idx, all_preds, average='macro', zero_division=0)
    print(f'Epoch {epoch+1}/{N_EPOCHS} | Loss: {total_loss/len(train_loader):.4f} | Valid F1 Product: {f1:.4f}', end='')

    if f1 > best_f1:
        best_f1    = f1
        best_state = copy.deepcopy(hier_model.state_dict())
        patience_cnt = 0
        
    else:
        patience_cnt += 1
        print(f' (patience {patience_cnt}/{patience})')
        if patience_cnt >= patience:
            print('Early stopping!')
            break

hier_model.load_state_dict(best_state)
print(f'\nΚαλύτερο F1 Product: {best_f1:.4f}')

In [ ]:
# Test predictions για Προσέγγιση Α
bert_test_hazard_probs = np.load('bertbase_focal_test_hazard_probs.npy')
pred_hazard_test_idx   = bert_test_hazard_probs.argmax(axis=1)
hazard_onehot_test     = np.eye(n_hazard)[pred_hazard_test_idx]

dummy_product = np.zeros(len(texts_test), dtype=int)
test_loader = DataLoader(
    HierarchicalDataset(texts_test, hazard_onehot_test, dummy_product, tokenizer, MAX_LENGTH),
    batch_size=BATCH_SIZE, shuffle=False
)

hier_model.eval()
test_product_preds = []
with torch.no_grad():
    for batch in test_loader:
        batch.pop('label')
        hazard_onehot = batch.pop('hazard_onehot').to(device)
        batch = {k: v.to(device) for k, v in batch.items()}
        preds = hier_model(**batch, hazard_onehot=hazard_onehot).argmax(dim=1).cpu().numpy()
        test_product_preds.extend(preds)

test_hazard_labels  = le_hazard.inverse_transform(pred_hazard_test_idx)
test_product_labels = le_product.inverse_transform(test_product_preds)

pd.DataFrame({
    'id': test['id'],
    'hazard-category': test_hazard_labels,
    'product-category': test_product_labels
}).to_csv('submission_hierarchical_a.csv', index=False)

# Validation score για Προσέγγιση Α
valid_product_preds_a = le_product.inverse_transform(all_preds)
pred_hazard_valid_str = le_hazard.inverse_transform(pred_hazard_valid_idx)

print('=== ΑΞΙΟΛΟΓΗΣΗ Προσέγγιση Α — Sequential BERT ===')
score_a = official_st1_score(
    valid['hazard-category'], pred_hazard_valid_str,
    valid['product-category'], valid_product_preds_a
)
print('\nΑποθηκεύτηκε: submission_hierarchical_a.csv')

In [ ]:
# Αποθήκευση hierarchical product probs για ensemble
hier_model.eval()
hier_test_product_probs = []
with torch.no_grad():
    for batch in test_loader:
        batch.pop('label')
        hazard_onehot = batch.pop('hazard_onehot').to(device)
        batch = {k: v.to(device) for k, v in batch.items()}
        probs = torch.softmax(hier_model(**batch, hazard_onehot=hazard_onehot), dim=1).cpu().numpy()
        hier_test_product_probs.append(probs)

hier_test_product_probs = np.vstack(hier_test_product_probs)
np.save('hierarchical_test_product_probs.npy', hier_test_product_probs)
print(f' Αποθηκεύτηκε: hierarchical_test_product_probs.npy  {hier_test_product_probs.shape}')

In [ ]:
print('\n' + '='*55)
print('ΣΥΓΚΡΙΣΗ HIERARCHICAL APPROACHES')
print('='*55)
print(f'Προσέγγιση Α (Sequential BERT):  {score_a:.4f}')
print(f'Προσέγγιση Β (Conditional SVM):  {score_b:.4f}')
print(f'\nBest so far (non-hierarchical):  0.8173')